In [2]:
import pandas as pd
import numpy as py

In [9]:
folder = r"data\sampled data"

orders_s = pd.read_csv(folder + r"\orders.csv")
customers_s = pd.read_csv(folder + r"\customers.csv")
order_items_s = pd.read_csv(folder + r"\order_items.csv")
payments_s    = pd.read_csv(folder + r"\payments.csv")
reviews_s     = pd.read_csv(folder + r"\reviews.csv")
products_s    = pd.read_csv(folder + r"\products.csv")
sellers_s     = pd.read_csv(folder + r"\sellers.csv")
category_tr_s = pd.read_csv(folder + r"\category_tr.csv")
geo_clean     = pd.read_csv(folder + r"\geolocation.csv")

print("All 9 tables loaded")

for n, d in [("orders",orders_s),("customers",customers_s),("order_items",order_items_s),
             ("payments",payments_s),("reviews",reviews_s),("products",products_s),
             ("sellers",sellers_s),("category_tr",category_tr_s),("geo_clean",geo_clean)]:
    print(f"{n:13s} → {d.shape[0]:>7,} rows | {d.shape[1]} cols")


All 9 tables loaded
orders        →  10,000 rows | 8 cols
customers     →  10,000 rows | 5 cols
order_items   →  11,383 rows | 7 cols
payments      →  10,476 rows | 5 cols
reviews       →   9,959 rows | 7 cols
products      →   6,747 rows | 9 cols
sellers       →   1,654 rows | 4 cols
category_tr   →      71 rows | 2 cols
geo_clean     →   6,486 rows | 5 cols


- order_items (11,383) outnumber orders (10,000) — at least 1,383 orders had more than one item in the cart.
- payments (10,476) exceed orders by 476, meaning a small but real share of customers split payment across multiple methods or installments.
- reviews (9,959) fall short by 41 — those orders completed delivery but never received any customer feedback.
- With 1,654 sellers covering 6,747 products, each seller lists roughly 4 products on average.
- category_tr is a clean 71-row lookup — every Portuguese category name maps to an English translation with no gaps.

In [29]:
orders_s.head(2)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,b9a6c5f5df52c7226ac85aee7524c27f,f160aaf480efdfa7268f0fa535f73e76,delivered,2018-06-12 20:07:44,2018-06-12 20:44:26,2018-06-13 13:09:00,2018-06-19 12:44:08,2018-07-17 00:00:00
1,261e71d2349c713eafa9f3df5972b95d,d6708bbbd2d419475869a84e41f620a1,delivered,2018-01-20 12:15:57,2018-01-20 12:37:13,2018-01-25 21:42:52,2018-01-30 11:32:35,2018-02-15 00:00:00


\\Build order_level table

In [31]:
pay_agg = (
    payments_s
    .groupby("order_id")
    .agg(
        total_payment_value = ("payment_value", "sum"),
        max_installments = ("payment_installments", "max"),
        n_payments_methods = ("payment_type", "nunique"),
        main_payment_type = ("payment_type", lambda x : x.mode().iloc[0]),
    )
    .reset_index()
)
pay_agg.head(3)

,order_id,total_payment_value,max_installments,n_payments_methods,main_payment_type
0,001ac194d4a326a6fa99b581e9a3d963,62.54,1,1,boleto
1,001e7cf2ad6bef3ade12ebc56ceaf0f3,51.10,2,1,credit_card
2,00259a44fcad3fc0474329e925d14fc3,34.09,1,1,credit_card


In [33]:
rev_agg = (
    reviews_s
    .groupby("order_id")
    .agg(
        avg_review_score = ("review_score", "mean"),
        n_reviews = ("review_id", "nunique"),
    )
    .reset_index()
)
rev_agg.head(3)

,order_id,avg_review_score,n_reviews
0,001ac194d4a326a6fa99b581e9a3d963,5.0,1
1,001e7cf2ad6bef3ade12ebc56ceaf0f3,1.0,1
2,00259a44fcad3fc0474329e925d14fc3,4.0,1


In [32]:
customers_s.head(3)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
1,5aa9e4fdd4dfd20959cad2d772509598,2a46fb94aef5cbeeb850418118cee090,20231,rio de janeiro,RJ
2,f681356046d9fde60e70c73a18d65ea2,5f102dd37243f152aec3607970aad100,9121,santo andre,SP


In [20]:
order_level = (
    orders_s
    .merge(customers_s, on="customer_id", how="left")
    .merge(pay_agg, on="order_id", how="left")
    .merge(rev_agg, on="order_id", how="left")
    .merge(geo_clean,
           left_on="customer_zip_code_prefix",
           right_on="geolocation_zip_code_prefix",
           how="left")
)

print("order_level shape:", order_level.shape)
print("Rows still 10,000?", order_level.shape[0] == 10000)
order_level.head(3)

order_level shape: (10000, 23)
Rows still 10,000? True


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,max_installments,n_payments_methods,main_payment_type,avg_review_score,n_reviews,geolocation_zip_code_prefix,geolocation_lat,gelocation_lng,geolocation_city,geolocation_state
0,b9a6c5f5df52c7226ac85aee7524c27f,f160aaf480efdfa7268f0fa535f73e76,delivered,2018-06-12 20:07:44,2018-06-12 20:44:26,2018-06-13 13:09:00,2018-06-19 12:44:08,2018-07-17 00:00:00,2d1bf256227e4d22d10ea6c0b81809d7,96900,...,2.0,1.0,credit_card,5.0,1.0,96900.0,-29.416784,-53.026586,sobradinho,RS
1,261e71d2349c713eafa9f3df5972b95d,d6708bbbd2d419475869a84e41f620a1,delivered,2018-01-20 12:15:57,2018-01-20 12:37:13,2018-01-25 21:42:52,2018-01-30 11:32:35,2018-02-15 00:00:00,12bf514b8d413d8cbe66a2665f4b724c,30840,...,10.0,1.0,credit_card,5.0,1.0,30840.0,-19.890278,-43.999366,belo horizonte,MG
2,67b50899f52995848c427e361e10dde3,1b353c00c71689afba44554e43cc5a76,delivered,2018-06-16 21:24:10,2018-06-16 21:36:59,2018-06-21 13:55:00,2018-06-27 13:17:27,2018-07-16 00:00:00,83c6df0d47130de38c99cebe96521e8a,80220,...,3.0,1.0,credit_card,1.0,1.0,80220.0,-25.456350,-49.265007,curitiba,PR


- All 10,000 orders survived four left joins — no rows dropped due to missing customer, payment, review, or geo records.
- The 23-column table brings purchase timestamps, delivery windows, payment totals, installment counts, review scores, and city-level coordinates into one flat structure ready for analysis.

\\Build item_level table

In [23]:
item_level = (
    order_items_s
    .merge(products_s, on="product_id", how="left")
    .merge(category_tr_s,
           on="product_category_name", how="left")
    .merge(sellers_s, on="seller_id", how="left")   
    .merge(orders_s[["order_id", "order_purchase_timestamp", "order_status"]],  #Getting_these_for_revenue_trend
           on="order_id", how="left")
)

item_level.head(3)
    

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,...,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,order_purchase_timestamp,order_status
0,001ac194d4a326a6fa99b581e9a3d963,1,dbaee28f4ee64465838a229582d77520,ffc470761de7d0232558ba5e786e57b7,2018-07-10 02:51:10,54.00,8.54,construcao_ferramentas_construcao,58.0,1178.0,...,418.0,19.0,6.0,13.0,construction_tools_construction,7091,guarulhos,SP,2018-07-04 11:39:11,delivered
1,001e7cf2ad6bef3ade12ebc56ceaf0f3,1,bdcf6a834e8faa30dac3886c7a58e92e,2a84855fd20af891be03bc5924d2b453,2018-05-22 10:59:50,35.90,15.20,beleza_saude,26.0,394.0,...,1614.0,31.0,16.0,28.0,health_beauty,30111,belo horizonte,MG,2018-05-19 10:29:23,delivered
2,00259a44fcad3fc0474329e925d14fc3,1,0c4d0a08f95c7b7dc5dc7402bfdafd4c,9f505651f4a6abe901a56cdc21508025,2018-01-03 17:59:35,19.99,14.10,informatica_acessorios,49.0,409.0,...,300.0,16.0,6.0,17.0,computers_accessories,4102,sao paulo,SP,2017-12-27 17:52:11,delivered


In [25]:
item_level["item_revenue"] = item_level["price"] + item_level["freight_value"]

print("item_level shape:", item_level.shape)
print("Rows still 11,383?", item_level.shape[0] == len(order_items_s))
item_level.head(3)

item_level shape: (11383, 22)
Rows still 11,383? True


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,...,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,order_purchase_timestamp,order_status,item_revenue
0,001ac194d4a326a6fa99b581e9a3d963,1,dbaee28f4ee64465838a229582d77520,ffc470761de7d0232558ba5e786e57b7,2018-07-10 02:51:10,54.00,8.54,construcao_ferramentas_construcao,58.0,1178.0,...,19.0,6.0,13.0,construction_tools_construction,7091,guarulhos,SP,2018-07-04 11:39:11,delivered,62.54
1,001e7cf2ad6bef3ade12ebc56ceaf0f3,1,bdcf6a834e8faa30dac3886c7a58e92e,2a84855fd20af891be03bc5924d2b453,2018-05-22 10:59:50,35.90,15.20,beleza_saude,26.0,394.0,...,31.0,16.0,28.0,health_beauty,30111,belo horizonte,MG,2018-05-19 10:29:23,delivered,51.10
2,00259a44fcad3fc0474329e925d14fc3,1,0c4d0a08f95c7b7dc5dc7402bfdafd4c,9f505651f4a6abe901a56cdc21508025,2018-01-03 17:59:35,19.99,14.10,informatica_acessorios,49.0,409.0,...,16.0,6.0,17.0,computers_accessories,4102,sao paulo,SP,2017-12-27 17:52:11,delivered,34.09


- All 11,383 item rows came through the five-table chain intact — no duplicates or drops introduced by any of the joins.
- item_revenue (price + freight) captures the true per-line cost and can be summed per order to cross-check against the payment totals in order_level.

In [27]:
order_level.to_csv(folder + r"\order_level.csv", index=False)
item_level.to_csv(folder + r"\item_level.csv", index=False)

print("Saved: order_level.csv, item_level.csv")

Saved: order_level.csv, item_level.csv
